# SENTINEL — EDA & Evaluation Notebook
Dark-style exploratory and model evaluation walkthrough.

In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

meter = pd.read_csv('../data/meter_readings.csv', parse_dates=['timestamp'])
meta = pd.read_csv('../data/meter_metadata.csv')
forecast = pd.read_csv('../data/forecast_results.csv', parse_dates=['timestamp'])
anomaly = pd.read_csv('../data/anomaly_results.csv')
with open('../data/anomaly_summary.json') as f:
    anomaly_summary = json.load(f)

meter['hour'] = meter['timestamp'].dt.hour
meter['dow'] = meter['timestamp'].dt.dayofweek
meter['date'] = meter['timestamp'].dt.date
meter.head()

## 1) Data Overview

In [ ]:
print('meter_readings shape:', meter.shape)
print('meter_metadata shape:', meta.shape)
print('\nDtypes:\n', meter.dtypes)
print('\nMissing values:\n', meter.isna().sum())
print('\nMeter count by locality:\n', meta['locality'].value_counts())
print('\nMeter count by category:\n', meta['category'].value_counts())

## 2) Consumption Pattern Analysis

In [ ]:
hourly_avg = meter.groupby('hour')['consumption_kwh'].mean()
daily_avg = meter.groupby('date')['consumption_kwh'].mean()
weekly_avg = meter.groupby('dow')['consumption_kwh'].mean()

fig, axes = plt.subplots(1,3, figsize=(18,4))
sns.lineplot(x=hourly_avg.index, y=hourly_avg.values, ax=axes[0], color='#00D4FF')
axes[0].set_title('Hourly Average Consumption')
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('kWh')

sns.lineplot(x=range(len(daily_avg)), y=daily_avg.values, ax=axes[1], color='#FF6B35')
axes[1].set_title('Daily Average Consumption')
axes[1].set_xlabel('Day Index'); axes[1].set_ylabel('kWh')

sns.barplot(x=weekly_avg.index, y=weekly_avg.values, ax=axes[2], color='#00E676')
axes[2].set_title('Weekly Pattern (0=Mon)')
axes[2].set_xlabel('Day of week'); axes[2].set_ylabel('kWh')
plt.tight_layout()

hm = meter.groupby(['dow','hour'])['consumption_kwh'].mean().unstack(0)
plt.figure(figsize=(10,5))
sns.heatmap(hm, cmap='mako')
plt.title('Consumption Heatmap (hour × day_of_week)')
plt.xlabel('Day of Week'); plt.ylabel('Hour')
plt.show()

## 3) Anomaly Distribution

In [ ]:
flags = anomaly[anomaly['flag_status']=='FLAGGED']

fig, axes = plt.subplots(1,2, figsize=(14,4))
sns.countplot(data=flags, x='anomaly_type', ax=axes[0], palette='rocket')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_title('Flagged Meter Count by Anomaly Type')

sns.histplot(flags['confidence_score'], bins=20, kde=True, ax=axes[1], color='#00D4FF')
axes[1].set_title('Confidence Score Distribution')
axes[1].set_xlabel('Confidence (%)')
plt.tight_layout(); plt.show()

## 4) Forecasting Evaluation

In [ ]:
metrics = pd.read_csv('../data/forecast_metrics.csv')
display(metrics)

sample_feeders = metrics['feeder_id'].head(3).tolist()
fig, axes = plt.subplots(len(sample_feeders), 1, figsize=(14, 4*len(sample_feeders)))
if len(sample_feeders)==1:
    axes=[axes]
for ax, fid in zip(axes, sample_feeders):
    sub = forecast[forecast['feeder_id']==fid].sort_values('timestamp').head(200)
    ax.plot(sub['timestamp'], sub['actual_kwh'], label='Actual', color='#00E676')
    ax.plot(sub['timestamp'], sub['predicted_kwh'], label='Predicted', color='#00D4FF')
    ax.plot(sub['timestamp'], sub['baseline_kwh'], label='Baseline', color='#6B7A9F')
    ax.set_title(f'Feeder {fid} — Forecast vs Actual vs Baseline')
    ax.legend()
plt.tight_layout(); plt.show()

## 5) Detection Evaluation

In [ ]:
print('Anomaly summary JSON:')
print(json.dumps(anomaly_summary, indent=2))

cm = anomaly_summary['confusion_matrix']
cm_arr = np.array([[cm['tn'], cm['fp']], [cm['fn'], cm['tp']]])
plt.figure(figsize=(4,4))
sns.heatmap(cm_arr, annot=True, fmt='d', cmap='magma', cbar=False)
plt.title('Confusion Matrix (Normal/Anomaly)')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

per_type = pd.DataFrame(anomaly_summary['per_type']).T
display(per_type)

y_true = (meter.groupby('meter_id')['is_anomaly'].max()>0).astype(int)
pred_idx = anomaly.set_index('meter_id')
y_score = pred_idx['confidence_score'].reindex(y_true.index).fillna(0)/100
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, color='#00D4FF', label=f'ROC AUC={roc_auc:.3f}')
plt.plot([0,1],[0,1], '--', color='gray')
plt.title('ROC Curve (Meter-level anomaly scoring)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(); plt.show()

## 6) False Positive Analysis
Look at normal meters with high confidence and compare to theft-like drops.

In [ ]:
normal_high = anomaly[(anomaly['flag_status']=='NORMAL')].sort_values('confidence_score', ascending=False).head(5)
display(normal_high[['meter_id','locality','confidence_score','deviation_pct','personal_baseline_kwh','peer_baseline_kwh']])

for mid in normal_high['meter_id'].tolist()[:3]:
    sub = meter[meter['meter_id']==mid].sort_values('timestamp').tail(96*7)
    plt.figure(figsize=(10,2.6))
    plt.plot(sub['timestamp'], sub['consumption_kwh'], color='#FF6B35')
    plt.title(f'Normal meter with theft-like patterns: {mid}')
    plt.xlabel('Timestamp'); plt.ylabel('kWh')
    plt.tight_layout(); plt.show()